# DDoS LSTM Anomaly Detection — CIC-IoT 2024
Run each cell top-to-bottom. Enable GPU: Runtime → Change runtime type → T4 GPU.

In [ ]:
# 0. Install dependencies
!pip install -q pandas scikit-learn tensorflow matplotlib seaborn

In [ ]:
# 1. Upload your DDoS CSV from CIC-IoT 2024
# Download from: https://www.unb.ca/cic/datasets/iotdataset-2024.html
from google.colab import files
uploaded = files.upload()   # pick your ddos.csv
DATA_PATH = list(uploaded.keys())[0]
print('Uploaded:', DATA_PATH)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
import tensorflow as tf
from tensorflow.keras import layers, callbacks
import matplotlib.pyplot as plt
import seaborn as sns

tf.random.set_seed(42)
np.random.seed(42)

FEATURES = [
    'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets',
    'Total Length of Fwd Packets', 'Total Length of Bwd Packets',
    'Fwd Packet Length Max', 'Fwd Packet Length Mean', 'Bwd Packet Length Mean',
    'Flow Bytes/s', 'Flow Packets/s', 'SYN Flag Count', 'ACK Flag Count',
    'Packet Length Mean', 'Packet Length Std', 'Average Packet Size',
]
LABEL_COL  = 'Label'
TIMESTEPS  = 20
STEP       = 10
THRESHOLD  = 0.5

In [ ]:
# 2. Load & clean
df = pd.read_csv(DATA_PATH, low_memory=False)
df.columns = df.columns.str.strip()
print('Raw shape:', df.shape)
print(df[LABEL_COL].value_counts())

available = [f for f in FEATURES if f in df.columns]
print('Using features:', available)

df = df[available + [LABEL_COL]].copy()
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df.dropna(inplace=True)

# Binary label: 0=benign, 1=attack
df['label_binary'] = (df[LABEL_COL].str.upper() != 'BENIGN').astype(int)
print('Clean shape:', df.shape)
print('Benign:', (df['label_binary']==0).sum(), ' Attack:', (df['label_binary']==1).sum())

In [ ]:
# 3. Scale features
scaler = MinMaxScaler()
df[available] = scaler.fit_transform(df[available])

In [ ]:
# 4. Build sliding-window sequences
X_raw = df[available].values
y_raw = df['label_binary'].values

Xs, ys = [], []
for i in range(0, len(X_raw) - TIMESTEPS, STEP):
    Xs.append(X_raw[i:i+TIMESTEPS])
    ys.append(int(y_raw[i:i+TIMESTEPS].max()))

X_seq = np.array(Xs)
y_seq = np.array(ys)
print('Sequences:', X_seq.shape, '  Labels:', y_seq.shape)

n = len(X_seq)
t1, t2 = int(n*0.70), int(n*0.85)
X_train, y_train = X_seq[:t1],  y_seq[:t1]
X_val,   y_val   = X_seq[t1:t2], y_seq[t1:t2]
X_test,  y_test  = X_seq[t2:],  y_seq[t2:]

In [ ]:
# 5. Build LSTM model
timesteps, n_features = X_train.shape[1], X_train.shape[2]

model = tf.keras.Sequential([
    layers.Input(shape=(timesteps, n_features)),
    layers.LSTM(64, return_sequences=True),
    layers.Dropout(0.3),
    layers.LSTM(32),
    layers.Dropout(0.3),
    layers.Dense(16, activation='relu'),
    layers.Dense(1,  activation='sigmoid'),
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy',
             tf.keras.metrics.Precision(name='precision'),
             tf.keras.metrics.Recall(name='recall'),
             tf.keras.metrics.AUC(name='auc')]
)
model.summary()

In [ ]:
# 6. Train
classes = np.unique(y_train)
weights = compute_class_weight('balanced', classes=classes, y=y_train)
class_weight = dict(zip(classes.astype(int), weights))
print('Class weights:', class_weight)

cb = [
    callbacks.EarlyStopping(monitor='val_auc', patience=5,
                            restore_best_weights=True, mode='max'),
    callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3),
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=50,
    batch_size=256,
    class_weight=class_weight,
    callbacks=cb,
    verbose=1,
)

In [ ]:
# 7. Evaluate
y_prob = model.predict(X_test, batch_size=256).flatten()
y_pred = (y_prob >= THRESHOLD).astype(int)

print(classification_report(y_test, y_pred, target_names=['Benign','Attack']))

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Training curves
axes[0].plot(history.history['loss'], label='train')
axes[0].plot(history.history['val_loss'], label='val')
axes[0].set_title('Loss'); axes[0].legend()

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[1],
            xticklabels=['Benign','Attack'], yticklabels=['Benign','Attack'])
axes[1].set_title('Confusion Matrix')

# ROC
fpr, tpr, _ = roc_curve(y_test, y_prob)
auc = roc_auc_score(y_test, y_prob)
axes[2].plot(fpr, tpr, label=f'AUC={auc:.4f}')
axes[2].plot([0,1],[0,1],'k--')
axes[2].set_title('ROC Curve'); axes[2].legend()

plt.tight_layout()
plt.show()

In [ ]:
# 8. Save model
model.save('ddos_lstm_model.keras')
print('Model saved as ddos_lstm_model.keras')